# 🌿 Rainforest Dashboard — Data Acquisition
Pulls annual PRODES deforestation data from TerraBrasilis and loads it into the Supabase `Rainforest` schema.

**Run once** to populate the database. Re-run to refresh (truncates and reloads).

## Prerequisites
- Set environment variable `SUPABASE_DB_PASSWORD` in Deepnote project settings
- Supabase schema `Rainforest` must exist (run `db/rainforest_schema.sql` first)

In [25]:
import requests
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values
import os

# Supabase direct DB connection (not PostgREST — needed for INSERT)
SUPABASE_HOST = "supabase.butscher.cloud"
SUPABASE_DB   = "postgres"
SUPABASE_USER = "postgres"
SUPABASE_PASS = os.environ["SUPABASE_PASSWORD"]
SUPABASE_PORT = 5433

print("✓ Config loaded")

✓ Config loaded


## Step 1 — Fetch PRODES data from TerraBrasilis WFS

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

WFS_BASE = (
    "http://terrabrasilis.dpi.inpe.br/geoserver/prodes-amazon-nb/ows"
    "?service=WFS&version=1.0.0&request=GetFeature"
    "&typeName=prodes-amazon-nb:yearly_deforestation_biome"
    "&outputFormat=application/json"
    "&propertyName=state,main_class,year,area_km"
    "&CQL_FILTER=main_class='desmatamento' AND year="  # year appended below
)

YEARS = list(range(2010, 2025))

def fetch_year(yr):
    url = WFS_BASE + str(yr)
    resp = requests.get(url, timeout=180)
    resp.raise_for_status()
    features = resp.json().get("features", [])
    rows = [f["properties"] for f in features]
    return yr, rows

all_rows = []
print("Fetching all years in parallel...")

with ThreadPoolExecutor(max_workers=8) as pool:
    futures = {pool.submit(fetch_year, yr): yr for yr in YEARS}
    for fut in as_completed(futures):
        yr = futures[fut]
        try:
            yr, rows = fut.result()
            all_rows.extend(rows)
            print(f"  {yr}: {len(rows):>6} polygons ✓")
        except Exception as e:
            print(f"  {yr}: ERROR — {e}")

df_raw = pd.DataFrame(all_rows)
print(f"\n✓ {len(df_raw)} rows total fetched")
print(f"Columns: {df_raw.columns.tolist()}")
df_raw.head(3)

## Step 2 — Transform
Inspect column names above and adjust the rename mapping if needed.

In [ ]:
# Rename, filter to desmatamento only, then AGGREGATE per year+state
df = df_raw.rename(columns={"state": "state_name", "area_km": "area_km2"}).copy()

# Keep only actual deforestation rows (not clouds, not forest baseline)
df = df[df["main_class"] == "desmatamento"].copy()

df["year"]     = df["year"].astype(int)
df["area_km2"] = df["area_km2"].astype(float)

# Aggregate: SUM area per year + state — this is what the dashboard needs
df = (
    df.groupby(["year", "state_name"], as_index=False)["area_km2"]
    .sum()
    .round(2)
)

# Use a single class label (all rows are desmatamento)
df["class_name"] = "Desmatamento"

print(f"✓ {len(df)} aggregated rows, years {df.year.min()}–{df.year.max()}")
print(df.groupby("year")["area_km2"].sum().to_string())
df.head(5)

## Step 3 — Calculate cumulative area

In [ ]:
df = df.sort_values(["state_name", "year"]).reset_index(drop=True)
df["accumulated_km2"] = (
    df.groupby("state_name")["area_km2"]
    .cumsum()
    .round(2)
)
print(f"✓ Cumulative area calculated")
df.head(9)  # erste 9 Zeilen = alle Staaten für Jahr 1

## Step 4 — Load into Supabase

In [28]:
conn = psycopg2.connect(
    host=SUPABASE_HOST, dbname=SUPABASE_DB,
    user=SUPABASE_USER, password=SUPABASE_PASS, port=SUPABASE_PORT
)
cur = conn.cursor()

cur.execute('TRUNCATE "Rainforest".fact_deforestation CASCADE')
cur.execute('TRUNCATE "Rainforest".dim_state CASCADE')
cur.execute('TRUNCATE "Rainforest".dim_class CASCADE')

# Insert dim_state
state_map = {}
for s in df["state_name"].unique():
    cur.execute(
        'INSERT INTO "Rainforest".dim_state (state_code, state_name) '
        'VALUES (%s, %s) RETURNING state_id',
        (s[:2].upper(), s)
    )
    state_map[s] = cur.fetchone()[0]

# Insert dim_class
class_map = {}
for c in df["class_name"].unique():
    cur.execute(
        'INSERT INTO "Rainforest".dim_class (class_name) '
        'VALUES (%s) RETURNING class_id',
        (c,)
    )
    class_map[c] = cur.fetchone()[0]

# Insert fact_deforestation
records = [
    (int(r.year), state_map[r.state_name], class_map[r.class_name],
     float(r.area_km2), float(r.accumulated_km2))
    for r in df.itertuples()
]
execute_values(
    cur,
    'INSERT INTO "Rainforest".fact_deforestation '
    '(year, state_id, class_id, area_km2, accumulated_km2) VALUES %s',
    records
)
conn.commit()
conn.close()

print(f"✓ {len(state_map)} states, {len(class_map)} classes, {len(records)} fact rows loaded")

✓ 8 states, 10 classes, 50000 fact rows loaded


## ✅ Done
Data is now in Supabase. Verify in the Supabase SQL Editor:
```sql
SELECT year, COUNT(*) as rows FROM "Rainforest".fact_deforestation GROUP BY year ORDER BY year;
```

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=65588d6b-f974-43a3-ad1e-5e3c5483491a' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>